### 2.1 理论计算题
序列：ababc，相邻转移对：$(a,b),(b,a),(a,b),(b,c)$
词汇表 $V=\{a,b,c\}$，词汇大小 $|V|=3$
拉普拉斯平滑（加1平滑）条件概率公式：
$$
p(y|x) = \frac{count(x\to y)+1}{count(x)+|V|}
$$

1. 统计 $x=\text{'b'}$ 的相关计数
$count(b)=2$（b作为前驱出现2次）
$count(b\to a)=1$（转移 b→a 出现1次）
$count(b\to c)=1$（转移 b→c 出现1次）

计算 $p(\text{'a'}|\text{'b'})$：
$$
p(a|b) = \frac{count(b\to a)+1}{count(b)+3}
= \frac{1+1}{2+3}
= \frac{2}{5}
$$

2. 计算 $p(\text{'c'}|\text{'b'})$：
$$
p(c|b) = \frac{count(b\to c)+1}{count(b)+3}
= \frac{1+1}{2+3}
= \frac{2}{5}
$$

### 2.2 编程题

In [2]:
import string
from collections import Counter

def preprocess_text(text, n):
    # 1. 转小写，去除标点，仅保留字母和空格
    text_lower = text.lower()
    # 去掉所有标点符号
    punc_set = set(string.punctuation)
    clean_chars = [c for c in text_lower if c not in punc_set]
    clean_text = ''.join(clean_chars)
    
    # 2. 按空格分词
    word_list = clean_text.split()
    # 过滤空字符串（多空格导致）
    word_list = [w for w in word_list if w]
    
    # 3. 构建词汇表：按频率降序，ID从0开始
    word_count = Counter(word_list)
    # 先按频次高到低，频次相同按单词字母序
    sorted_words = sorted(word_count.keys(), key=lambda x: (-word_count[x], x))
    word2id = {word: idx for idx, word in enumerate(sorted_words)}
    
    # 4. 滑动窗口生成特征序列与标签，无后续词则丢弃
    feature_list = []
    label_list = []
    seq_len = len(word_list)
    for i in range(seq_len - n):
        window = word_list[i:i+n]
        next_word = word_list[i + n]
        feature_list.append(window)
        label_list.append(next_word)
    
    return word2id, (feature_list, label_list)

#  测试
if __name__ == "__main__":
    test_input = "The time machine"
    window_n = 2
    vocab, (features, labels) = preprocess_text(test_input, window_n)
    
    print("词汇表 word2id：")
    print(vocab)
    print("\n特征序列列表：")
    print(features)
    print("\n对应标签列表：")
    print(labels)

词汇表 word2id：
{'machine': 0, 'the': 1, 'time': 2}

特征序列列表：
[['the', 'time']]

对应标签列表：
['machine']


### 3.1 理论计算题
已知
$$h_t = W_{hh}h_{t-1} + W_{hx}x_t$$
$$o_t = W_{oh}h_t$$
$$L = \frac{1}{2}\sum_{t=1}^T (o_t - y_t)^2$$

1. 求单步损失对输出的梯度
单步损失 $L_t = \frac{1}{2}(o_t-y_t)^2$
$$\frac{\partial L_t}{\partial o_t} = o_t - y_t$$

2. 输出对当前隐状态的梯度
$$\frac{\partial o_t}{\partial h_t} = W_{oh}^\top$$
$$\frac{\partial L_t}{\partial h_t} = (o_t - y_t)W_{oh}^\top$$

3. 隐状态时序链式求导关系
由 $h_t$ 的递推式可得
$$\frac{\partial h_t}{\partial h_{t-1}} = W_{hh}$$
对任意时刻 $k$，损失对隐状态的递推关系为
$$\frac{\partial L}{\partial h_k} = \frac{\partial L_k}{\partial h_k} + \frac{\partial L}{\partial h_{k+1}}W_{hh}$$

4. 推导损失对 $W_{hh}$ 的整体梯度
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=2}^T \frac{\partial L}{\partial h_t} \cdot h_{t-1}^\top$$
将 $\frac{\partial L}{\partial h_t}$ 沿时间链全部展开
$$\frac{\partial L}{\partial h_t} = \sum_{k=t}^T (o_k - y_k)W_{oh}^\top W_{hh}^{k-t}$$
合并得到完整梯度表达式
$$\frac{\partial L}{\partial W_{hh}} = \sum_{t=2}^T \left( \sum_{k=t}^T (o_k - y_k)W_{oh}^\top W_{hh}^{k-t} \right) h_{t-1}^\top$$

梯度消失、爆炸条件分析
设 $W_{hh}$ 最大特征值的绝对值为 $\lambda$，梯度中包含 $W_{hh}$ 的幂次连乘项。

若 $\lambda < 1$：时间间隔越大，$W_{hh}^{k-t}$ 数值指数衰减，远距离时序梯度趋近于0，出现梯度消失。

若 $\lambda > 1$：时间间隔越大，$W_{hh}^{k-t}$ 数值指数放大，梯度数值急剧增大，出现梯度爆炸。

### 3.2 编程题

In [1]:
import numpy as np

class SimpleRNNCell:
    def forward(self, x_t, h_prev, W_hx, W_hh, b_h):
        # 前向传播，计算当前隐状态h_t
        z = np.matmul(x_t, W_hx) + np.matmul(h_prev, W_hh) + b_h
        h_t = np.tanh(z)
        # 缓存前向中间变量，反向传播使用
        self.cache = (z, x_t, h_prev, W_hx, W_hh)
        return h_t

    def backward(self, dh_next):
        # 反向传播，输入上游梯度dh_next = dL/dh_t
        z, x_t, h_prev, W_hx, W_hh = self.cache
        batch_size = x_t.shape[0]

        # tanh导数：d(tanh(z))/dz = 1 - tanh(z)^2
        dz = dh_next * (1 - np.tanh(z) ** 2)

        # 各参数梯度
        dx_t = np.matmul(dz, W_hx.T)
        dh_prev = np.matmul(dz, W_hh.T)
        dW_hx = np.matmul(x_t.T, dz)
        dW_hh = np.matmul(h_prev.T, dz)
        db_h = np.sum(dz, axis=0, keepdims=True)

        return dx_t, dh_prev, dW_hx, dW_hh, db_h

# ---------------------- 测试代码，输出梯度结果 ----------------------
if __name__ == "__main__":
    # 超参数
    batch_size = 2
    input_size = 3
    hidden_size = 4

    # 随机初始化输入、隐状态、权重
    x_t = np.random.randn(batch_size, input_size)
    h_prev = np.random.randn(batch_size, hidden_size)
    W_hx = np.random.randn(input_size, hidden_size)
    W_hh = np.random.randn(hidden_size, hidden_size)
    b_h = np.random.randn(1, hidden_size)

    rnn_cell = SimpleRNNCell()
    # 前向
    h_t = rnn_cell.forward(x_t, h_prev, W_hx, W_hh, b_h)
    print("前向传播输出 h_t 形状：", h_t.shape)

    # 模拟上游梯度 dL/dh_t
    dh_next = np.random.randn(batch_size, hidden_size)
    dx_t, dh_prev, dW_hx, dW_hh, db_h = rnn_cell.backward(dh_next)

    print("\n反向传播各梯度形状：")
    print("dx_t shape:", dx_t.shape)
    print("dh_prev shape:", dh_prev.shape)
    print("dW_hx shape:", dW_hx.shape)
    print("dW_hh shape:", dW_hh.shape)
    print("db_h shape:", db_h.shape)

前向传播输出 h_t 形状： (2, 4)

反向传播各梯度形状：
dx_t shape: (2, 3)
dh_prev shape: (2, 4)
dW_hx shape: (3, 4)
dW_hh shape: (4, 4)
db_h shape: (1, 4)


### 4.1 理论计算题
单层单向RNN单元参数（单方向，输入维度$in$，隐藏维度$H$）
权重：$in \times H + H \times H$
偏置：$H$
单层单向总参数：$H(in + H + 1)$

双向RNN包含前向、后向两个独立单向单元，单层双向参数：
$2H(in + H + 1)$

1. 第1层双向RNN
输入维度为$D$，单层双向参数：
$2H(D + H + 1)$

2. 第2~L层双向RNN
上一层双向输出拼接维度为$2H$，每一层参数：
$2H(2H + H + 1) = 2H(3H + 1)$
共$L-1$层，总和：
$(L-1) \cdot 2H(3H + 1)$

3. 最后输出全连接层
输入为顶层双向拼接特征$2H$，输出维度$O$
权重：$2H \cdot O$
偏置：$O$
输出层总参数：$2HO + O$

4. 全部参数总和表达式
$$
\begin{aligned}
Total &= 2H(D+H+1) + 2H(3H+1)(L-1) + 2HO + O
\end{aligned}
$$

展开整理
$$
Total = 2HD + 2H^2 + 2H + 2(L-1)(3H^2+H) + O(2H+1)
$$

### 4.2 编程题


In [2]:
import torch
import torch.nn as nn

class BiRNNEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super().__init__()
        self.rnn = nn.RNN(
            input_size=input_dim,
            hidden_size=hidden_dim,
            bidirectional=True,
            batch_first=False
        )

    def forward(self, X):
        # X: (seq_len, batch, input_dim)
        seq_out, h_n = self.rnn(X)
        # seq_out 已经是前向+后向拼接 (seq_len, batch, 2*hidden_dim)
        # h_n shape: (2, batch, hidden_dim)
        forward_last = h_n[0]
        backward_last = h_n[1]
        final_hidden = torch.cat([forward_last, backward_last], dim=-1)
        return seq_out, final_hidden

# 测试代码
if __name__ == "__main__":
    seq_len = 10
    batch = 4
    input_dim = 8
    hidden_dim = 16

    encoder = BiRNNEncoder(input_dim, hidden_dim)
    X = torch.randn(seq_len, batch, input_dim)
    seq_output, final_state = encoder(X)

    print("各时间步拼接隐状态 shape:", seq_output.shape)
    print("最终拼接隐状态 shape:", final_state.shape)

各时间步拼接隐状态 shape: torch.Size([10, 4, 32])
最终拼接隐状态 shape: torch.Size([4, 32])


### 5.1 理论计算题
1. 目标函数推导
对于中心词 $w_c$，上下文正样本词 $w_o$，$K$ 个负样本 $w_{n_1},w_{n_2},...,w_{n_K}$
输入词向量 $\boldsymbol{v}_c$，输出向量：正样本 $\boldsymbol{u}_o$，负样本 $\boldsymbol{u}_{n_k}$

单个中心词-上下文对的对数似然目标：
$$
\log \sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o) + \sum_{k=1}^K \log \sigma(-\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k})
$$
其中 $\sigma(x) = \frac{1}{1+e^{-x}}$ 为sigmoid函数。

完整目标函数（最大化该值，训练损失取负）：
$$
\mathcal{L} = - \left[ \log \sigma(\boldsymbol{v}_c^\top \boldsymbol{u}_o) + \sum_{k=1}^K \log \sigma(-\boldsymbol{v}_c^\top \boldsymbol{u}_{n_k}) \right]
$$

2. 负样本采样方式
负样本从噪声分布 $P_n(w)$ 中采样，标准Skip-gram负采样采用词汇频次的3/4次幂分布：
$$
P_n(w) = \frac{count(w)^{3/4}}{\sum_{w'} count(w')^{3/4}}
$$
采样规则:  
每次抽取 $K$ 个单词作为负样本；  
采样时排除当前正上下文词 $w_o$，保证正负样本不重复；  
高频词更容易被抽到作为负样本，降低高频词过度占据训练权重的问题。

### 5.2 编程题

In [4]:
import torch
import torch.nn.functional as F

def cbow_forward_loss(context_indices, target_labels, W, W_out):
    """
    CBOW前向传播+完整softmax交叉熵损失计算
    参数：
        context_indices: 一批上下文词索引，shape [batch_size, context_size]
        target_labels: 一批中心词索引，shape [batch_size]
        W: 输入嵌入权重矩阵，shape (V, d)
        W_out: 输出权重矩阵，shape (d, V)
    返回：
        批次平均交叉熵损失标量
    """
    # 1. 取出所有上下文词对应的嵌入向量
    # context_embeds: [batch, context_size, d]
    context_embeds = W[context_indices]

    # 2. 计算平均上下文向量作为隐藏层 h
    h = torch.mean(context_embeds, dim=1)  # [batch, d]

    # 3. 计算全词汇logits，完整softmax
    logits = torch.matmul(h, W_out)  # [batch, V]

    # 4. 完整softmax交叉熵损失，无负采样
    loss = F.cross_entropy(logits, target_labels)
    return loss

# 测试
if __name__ == "__main__":
    # 超参数设定
    V = 15       # 词汇表大小
    d = 6        # 嵌入维度
    batch_size = 4
    context_size = 3

    # 初始化权重矩阵
    W = torch.randn(V, d)
    W_out = torch.randn(d, V)

    # 构造输入：每个样本3个上下文词索引
    batch_context = torch.tensor([
        [1, 2, 4],
        [0, 3, 5],
        [2, 6, 7],
        [1, 5, 8]
    ])
    # 对应每个样本的中心词目标索引
    batch_targets = torch.tensor([3, 6, 5, 9])

    # 计算损失
    loss_result = cbow_forward_loss(batch_context, batch_targets, W, W_out)

    # 打印完整输出信息
    print(f"词汇表大小 V = {V}, 嵌入维度 d = {d}")
    print(f"批次大小 batch_size = {batch_size}, 单侧上下文窗口 context_size = {context_size}")
    print(f"输入嵌入矩阵 W shape: {W.shape}")
    print(f"输出权重矩阵 W_out shape: {W_out.shape}")
    print(f"批次上下文索引矩阵 shape: {batch_context.shape}")
    print(f"批次目标中心词索引 shape: {batch_targets.shape}")
    print(f"\n批次平均交叉熵损失值：{loss_result.item():.4f}")

词汇表大小 V = 15, 嵌入维度 d = 6
批次大小 batch_size = 4, 单侧上下文窗口 context_size = 3
输入嵌入矩阵 W shape: torch.Size([15, 6])
输出权重矩阵 W_out shape: torch.Size([6, 15])
批次上下文索引矩阵 shape: torch.Size([4, 3])
批次目标中心词索引 shape: torch.Size([4])

批次平均交叉熵损失值：4.8453


### 6.1 理论计算题
已知
$Q\in \mathbb{R}^{2\times 4},\quad K\in \mathbb{R}^{3\times 4},\quad V\in \mathbb{R}^{3\times 5},\quad d_k=4,\sqrt{d_k}=2$

1：计算原始打分矩阵 $QK^\top$
$Q$ 形状 $2\times4$，$K^\top$ 形状 $4\times3$
$QK^\top \in \mathbb{R}^{2\times 3}$

2：缩放打分
$$
Score = \frac{QK^\top}{\sqrt{d_k}} = \frac{QK^\top}{2},\quad Score\in\mathbb{R}^{2\times 3}
$$

3：对每行做softmax得到注意力权重矩阵$Attn\_weight$
$Attn\_weight = \text{softmax}(Score),\quad Attn\_weight\in\mathbb{R}^{2\times 3}$
softmax按行操作，每行三个元素和为1。

4：加权求和得到注意力输出
$Attn\_out = Attn\_weight \cdot V$
$Attn\_weight$ 为 $2\times3$，$V$ 为 $3\times5$
最终输出矩阵维度：$\boldsymbol{\mathbb{R}^{2\times 5}}$

完整公式
$$
\text{Attention}(Q,K,V) = \text{softmax}\left(\frac{QK^\top}{\sqrt{d_k}}\right)V
$$

### 6.2 编程题

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

# 固定超参数
NUM_HEADS = 2
D_MODEL = 4
D_K = D_V = D_MODEL // NUM_HEADS  # d_k = d_v = 2

class MultiHeadAttention(nn.Module):
    def __init__(self):
        super().__init__()
        # 投影层：统一投影到 num_heads * d_k 维度
        self.w_q = nn.Linear(D_MODEL, NUM_HEADS * D_K)
        self.w_k = nn.Linear(D_MODEL, NUM_HEADS * D_K)
        self.w_v = nn.Linear(D_MODEL, NUM_HEADS * D_V)
        # 输出融合线性层
        self.w_o = nn.Linear(NUM_HEADS * D_V, D_MODEL)

    def scaled_dot_product_attention(self, q, k, v):
        # q,k,v: (num_heads, batch, seq_len, d_k)
        attn_score = torch.matmul(q, k.transpose(-2, -1)) / torch.sqrt(torch.tensor(D_K, dtype=torch.float32))
        attn_weight = F.softmax(attn_score, dim=-1)
        attn_out = torch.matmul(attn_weight, v)
        return attn_out

    def forward(self, X):
        # X shape: (seq_len, batch, d_model)
        seq_len, batch, _ = X.shape

        # 1. 线性投影 Q,K,V
        Q = self.w_q(X)
        K = self.w_k(X)
        V = self.w_v(X)

        # 2. 分头：(seq_len, batch, num_heads, d_k) -> (num_heads, batch, seq_len, d_k)
        Q = Q.view(seq_len, batch, NUM_HEADS, D_K).permute(2, 1, 0, 3)
        K = K.view(seq_len, batch, NUM_HEADS, D_K).permute(2, 1, 0, 3)
        V = V.view(seq_len, batch, NUM_HEADS, D_V).permute(2, 1, 0, 3)

        # 3. 每个头计算缩放点积注意力
        head_out = self.scaled_dot_product_attention(Q, K, V)
        # head_out: (num_heads, batch, seq_len, d_v)

        # 4. 拼接多头输出
        head_out = head_out.permute(2, 1, 0, 3)  # (seq_len, batch, num_heads, d_v)
        concat = head_out.contiguous().view(seq_len, batch, NUM_HEADS * D_V)

        # 5. 最终线性层，输出shape与输入一致
        output = self.w_o(concat)
        return output

# 测试
if __name__ == "__main__":
    # 构造输入 X: (seq_len, batch, d_model)
    seq_len = 5
    batch = 3
    X = torch.randn(seq_len, batch, D_MODEL)

    mha = MultiHeadAttention()
    out = mha(X)

    print(f"固定参数 num_heads={NUM_HEADS}, d_model={D_MODEL}, d_k={D_K}")
    print(f"输入X shape: {X.shape}")
    print(f"MHA输出 shape: {out.shape}")
    print(f"输出与输入形状是否相同: {out.shape == X.shape}")
    print(f"\n输出张量前两行数值：\n{out[:2]}")

固定参数 num_heads=2, d_model=4, d_k=2
输入X shape: torch.Size([5, 3, 4])
MHA输出 shape: torch.Size([5, 3, 4])
输出与输入形状是否相同: True

输出张量前两行数值：
tensor([[[-0.4370, -0.4278,  0.2799,  0.3057],
         [-0.4917, -0.5912,  0.3296,  0.3341],
         [-0.3696, -0.2282,  0.1976,  0.2765]],

        [[-0.4737, -0.4765,  0.3275,  0.3125],
         [-0.5104, -0.5378,  0.3622,  0.3273],
         [-0.3527, -0.2149,  0.1695,  0.2874]]], grad_fn=<SliceBackward0>)
